# Tweety-11 — Inférence causale & do-calculus (twin C# from-scratch)

Ce notebook est le **jumeau C# / .NET** de [Tweety-11-Causal.ipynb](Tweety-11-Causal.ipynb) — marathon de parité #4956.

La version Python s'appuie sur **Tweety `causal-1.30.jar`** (via JPype) : un reasoner causal **argumentatif** construit sur
des *Structural Causal Models* booléens. Ce twin C# **réimplémente le moteur causal from-scratch** (BCL .NET 9, 0 NuGet)
pour rendre **visibles les internes** : la chirurgie de graphe du do-operator, l'énumération des mondes possibles,
la sémantique contrefactuelle. La lib Java Tweety = `RECOVERABLE-MACHINE` (Python-only) ; le moteur from-scratch EST la substance.

**Pourquoi from-scratch ?** L'enjeu pédagogique de l'inférence causale (Pearl) n'est pas d'appeler une boîte noire,
mais de *voir* pourquoi `P(rain | observe drops) != P(rain | do drops)`. Cela exige d'exposer le graphe causal orienté,
la surgery `do(X)` qui rompt les liens entrants, et l'évaluation des mondes possibles.

## 1. Moteur causal — modèles structurels booléens

Un **Structural Causal Model (SCM)** booléen est un DAG d'atomes propositionnels :
- les **atomes de fond** (exogènes) sont les causes racines, libres ;
- les **atomes expliqués** (endogènes) sont définis par une **équation booléenne** sur leurs parents.

Étant donné une assignation des atomes de fond, tous les atomes expliqués sont **déterminés** par propagation
(le graphe est acyclique). Un *monde possible* = une assignation complète cohérente avec les équations.

In [1]:
using System;
using System.Linq;
using System.Text;
using System.Collections.Generic;

static void Show(string s) { s.Display(); }
Show("Moteur d'inference causale (Boolean structural causal models, from-scratch).");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Moteur d'inference causale (Boolean structural causal models, from-scratch).

In [2]:
#nullable enable
// --- Formule booleenne sur des atomes : || && ! ( ) + (vrai) - (faux) ---
public sealed class Formula
{
    public Func<IReadOnlyDictionary<string,bool>, bool> Eval;
    public HashSet<string> Vars;
    public Formula(Func<IReadOnlyDictionary<string,bool>, bool> e, HashSet<string> v) { Eval = e; Vars = v; }
    public static Formula Atom(string n) => new(a => a[n], new() { n });
    public static Formula Const(bool b) => new(_ => b, new());
    public static Formula Not(Formula f) => new(a => !f.Eval(a), f.Vars);
    public static Formula And(Formula x, Formula y) => new(a => x.Eval(a) && y.Eval(a), Union(x.Vars, y.Vars));
    public static Formula Or(Formula x, Formula y)  => new(a => x.Eval(a) || y.Eval(a), Union(x.Vars, y.Vars));
    static HashSet<string> Union(HashSet<string> a, HashSet<string> b) { var s = new HashSet<string>(a); s.UnionWith(b); return s; }
}

// Petite analyse recursive descendante (tokeniseur + grammaire PL minimale).
public static class FormulaParser
{
    public static Formula Parse(string src)
    {
        var toks = Tokenize(src);
        int i = 0;
        Formula f = ParseOr(toks, ref i);
        return f;
    }
    static List<string> Tokenize(string s)
    {
        var t = new List<string>();
        int i = 0;
        while (i < s.Length)
        {
            char c = s[i];
            if (char.IsWhiteSpace(c)) { i++; continue; }
            if (c == '(' || c == ')' || c == '+' || c == '-' || c == '!') { t.Add(c.ToString()); i++; continue; }
            if (c == '&' && i + 1 < s.Length && s[i + 1] == '&') { t.Add("&&"); i += 2; continue; }
            if (c == '|' && i + 1 < s.Length && s[i + 1] == '|') { t.Add("||"); i += 2; continue; }
            if (char.IsLetter(c) || c == '_')
            {
                int j = i;
                while (j < s.Length && (char.IsLetterOrDigit(s[j]) || s[j] == '_')) j++;
                t.Add(s.Substring(i, j - i)); i = j; continue;
            }
            throw new ArgumentException($"Caractere inattendu '{c}' dans la formule : {s}");
        }
        return t;
    }
    static Formula ParseOr(List<string> t, ref int i)
    {
        Formula f = ParseAnd(t, ref i);
        while (i < t.Count && t[i] == "||") { i++; f = Formula.Or(f, ParseAnd(t, ref i)); }
        return f;
    }
    static Formula ParseAnd(List<string> t, ref int i)
    {
        Formula f = ParseNot(t, ref i);
        while (i < t.Count && t[i] == "&&") { i++; f = Formula.And(f, ParseNot(t, ref i)); }
        return f;
    }
    static Formula ParseNot(List<string> t, ref int i)
    {
        if (i < t.Count && t[i] == "!") { i++; return Formula.Not(ParseNot(t, ref i)); }
        return ParsePrimary(t, ref i);
    }
    static Formula ParsePrimary(List<string> t, ref int i)
    {
        string tok = t[i];
        if (tok == "+") { i++; return Formula.Const(true); }
        if (tok == "-") { i++; return Formula.Const(false); }
        if (tok == "(") { i++; Formula f = ParseOr(t, ref i); i++; return f; } // saute ')'
        i++; return Formula.Atom(tok); // identificateur
    }
}

// --- Modele causal structurel booleen ---
public sealed class StructuralCausalModel
{
    public enum Kind { Background, Explainable }
    public Dictionary<string, Kind> Kinds = new();
    public Dictionary<string, Formula> Eq = new();   // eq d'evaluation
    public Dictionary<string, string> EqSrc = new(); // eq source pour affichage

    public List<string> Background => Kinds.Where(kv => kv.Value == Kind.Background).Select(kv => kv.Key).ToList();
    public List<string> Explainable => Kinds.Where(kv => kv.Value == Kind.Explainable).Select(kv => kv.Key).ToList();

    public void AddBackgroundAtom(string a) { Kinds[a] = Kind.Background; }
    public void AddExplainableAtom(string a, string src) { Kinds[a] = Kind.Explainable; Eq[a] = FormulaParser.Parse(src); EqSrc[a] = src; }

    // Ordre topologique sur les atomes expliques (Kahn ; les atomes de fond sont des sources).
    List<string> TopoOrder()
    {
        var remaining = Explainable.ToHashSet();
        var order = new List<string>();
        bool progress = true;
        while (remaining.Count > 0)
        {
            progress = false;
            foreach (var a in remaining.ToList())
            {
                bool depsOk = Eq[a].Vars.All(d => !remaining.Contains(d));
                if (depsOk) { order.Add(a); remaining.Remove(a); progress = true; }
            }
            if (!progress) throw new InvalidOperationException("SCM cyclique (equations structurelles doivent etre acycliques).");
        }
        return order;
    }

    // Propage : etant donne les atomes de fond, calcule le monde complet.
    public Dictionary<string, bool> Propagate(Dictionary<string, bool> bg)
    {
        var w = new Dictionary<string, bool>(bg);
        foreach (var a in TopoOrder()) w[a] = Eq[a].Eval(w);
        return w;
    }

    // Tous les mondes possibles = produit de toutes les assignations des atomes de fond.
    public List<Dictionary<string, bool>> AllWorlds()
    {
        var bg = Background;
        var worlds = new List<Dictionary<string, bool>>();
        int n = bg.Count;
        for (int mask = 0; mask < (1 << n); mask++)
        {
            var assign = new Dictionary<string, bool>();
            for (int i = 0; i < n; i++) assign[bg[i]] = ((mask >> i) & 1) == 1;
            worlds.Add(Propagate(assign));
        }
        return worlds;
    }

    public Dictionary<string, bool> BackgroundOf(Dictionary<string, bool> world)
    {
        var b = new Dictionary<string, bool>();
        foreach (var a in Background) b[a] = world[a];
        return b;
    }

    // Chirurgie du do-operator : do(X = val) rompt les liens entrants vers X (X devient constante).
    public StructuralCausalModel Intervene(string atom, bool val)
    {
        var c = Clone();
        c.Kinds[atom] = Kind.Explainable;   // n'est plus libre : fixe par l'intervention
        c.Eq[atom] = Formula.Const(val);
        c.EqSrc[atom] = val ? "+" : "-";
        return c;
    }

    public StructuralCausalModel Clone()
    {
        var c = new StructuralCausalModel();
        foreach (var kv in Kinds) c.Kinds[kv.Key] = kv.Value;
        foreach (var kv in Eq) c.Eq[kv.Key] = kv.Value;
        foreach (var kv in EqSrc) c.EqSrc[kv.Key] = kv.Value;
        return c;
    }

    public string PrettyPrint()
    {
        var sb = new StringBuilder();
        foreach (var a in Background) sb.AppendLine($"  [fond]      {a}");
        foreach (var a in Explainable) sb.AppendLine($"  [explique]  {a} <=> {EqSrc[a]}");
        return sb.ToString().TrimEnd();
    }
}

Show("StructuralCausalModel + parser de formules + propagation + Intervene (do-operator) prets.");


StructuralCausalModel + parser de formules + propagation + Intervene (do-operator) prets.

## 2. Reasoner causal — observation, intervention, contrefactuel

Trois types de requetes causales (Pearl) :

- **Observation** `P(Y | observe X)` : correlation. Parmi les mondes ou X est vrai, Y est-il etabli ?
- **Intervention** `P(Y | do X)` : on applique la chirurgie `do(X)` puis on mesure Y. Casse les confondeurs.
- **Contrefactuel** `observe W ; si X avait ete autre, Y aurait-il eu lieu ?` : monde reel + monde contrefactuel.

> Convention : une requete renvoie **True** si la conclusion est **etablie dans tous les mondes consistants**
> (si elle est vraie dans certains et fausse dans d'autres, la reponse est **False** = non determine).

In [3]:
#nullable enable
// Base de connaissance causale : enveloppe le SCM (tous les mondes du modele sont candidats).
public sealed class CausalKnowledgeBase
{
    public StructuralCausalModel Model;
    public CausalKnowledgeBase(StructuralCausalModel m) { Model = m; }
    public List<Dictionary<string,bool>> Worlds() => Model.AllWorlds();
}

// Declaration d'intervention : do(atom=val) pour plusieurs atomes + conclusion.
public sealed class InterventionalStatement
{
    public List<(string Atom, bool Val)> Interventions = new();
    public string Conclusion = "";
    public void AddIntervention(string atom, bool val) => Interventions.Add((atom, val));
    public void SetConclusion(string c) => Conclusion = c;
}

// Declaration contrefactuelle : observations (monde reel) + interventions contrefactuelles + conclusion.
public sealed class CounterfactualStatement
{
    public List<(string Atom, bool Val)> Observations = new();
    public List<(string Atom, bool Val)> CfInterventions = new();
    public string Conclusion = "";
    public void AddObservation(string atom, bool val) => Observations.Add((atom, val));
    public void AddCounterfactualIntervention(string atom, bool val) => CfInterventions.Add((atom, val));
    public void SetConclusion(string c) => Conclusion = c;
}

public sealed class CausalReasoner
{
    // Observation : conclusion etablie dans tous les mondes ou les atomes observes ont la valeur donnee.
    public bool Query(CausalKnowledgeBase ckb, List<(string Atom, bool Val)> observed, string conclusion)
    {
        var consistent = ckb.Worlds().Where(w => observed.All(o => w[o.Atom] == o.Val)).ToList();
        if (consistent.Count == 0) return false;
        return consistent.All(w => w[conclusion]);
    }
    // Raccourci : observer un atome = True.
    public bool Query(CausalKnowledgeBase ckb, List<string> observedTrue, string conclusion)
        => Query(ckb, observedTrue.Select(a => (a, true)).ToList(), conclusion);

    // Intervention : on applique do(*) puis on demande si la conclusion est etablie dans tous les mondes.
    public bool Query(CausalKnowledgeBase ckb, InterventionalStatement stmt)
    {
        var m = ckb.Model;
        foreach (var iv in stmt.Interventions) m = m.Intervene(iv.Atom, iv.Val);
        var worlds = m.AllWorlds();
        if (worlds.Count == 0) return false;
        return worlds.All(w => w[stmt.Conclusion]);
    }

    // Contrefactuel : mondes reels consistants avec l'observation ; sur CHACUN (fond fixe) on applique
    // l'intervention contrefactuelle et on re-evalue. La conclusion doit tenir dans tous les mondes contrefactuels.
    public bool Query(CausalKnowledgeBase ckb, CounterfactualStatement stmt)
    {
        var actual = ckb.Worlds().Where(w => stmt.Observations.All(o => w[o.Atom] == o.Val)).ToList();
        if (actual.Count == 0) return false;
        foreach (var w in actual)
        {
            var m = ckb.Model;
            foreach (var ci in stmt.CfInterventions) m = m.Intervene(ci.Atom, ci.Val);
            var cfWorld = m.Propagate(m.BackgroundOf(w));  // fond du monde reel, puis surgery
            if (!cfWorld[stmt.Conclusion]) return false;
        }
        return true;
    }
}

Show("CausalKnowledgeBase + InterventionalStatement + CounterfactualStatement + CausalReasoner prets.");


CausalKnowledgeBase + InterventionalStatement + CounterfactualStatement + CausalReasoner prets.

## 3. Le baromètre — corrélation n'est pas causalité

Exemple canonique : un baromètre qui tombe (`drops`) et la pluie (`rain`) sont **toutes deux causées** par l'orage (`storm`).
Observer le baromètre prédit la pluie (corrélation). **Forcer** le baromètre ne fait PAS pleuvoir (pas de lien causal `drops -> rain`).

> C'est la distinction fondamentale `P(rain | drops) != P(rain | do drops)`.

In [4]:
var storm = "storm"; var drops = "drops"; var rain = "rain";

var barometer = new StructuralCausalModel();
barometer.AddBackgroundAtom(storm);            // storm = cause racine exogene
barometer.AddExplainableAtom(drops, "storm");  // drops  <=> storm
barometer.AddExplainableAtom(rain,  "storm");  // rain   <=> storm

Show("SCM barometre :");
Show(barometer.PrettyPrint());


SCM barometre :

  [fond]      storm
  [explique]  drops <=> storm
  [explique]  rain <=> storm

In [5]:
var ckb_bar = new CausalKnowledgeBase(barometer);
var reasoner = new CausalReasoner();

// NIVEAU 1 : observation (correlation)
bool p_rain_if_drops = reasoner.Query(ckb_bar, new List<string> { drops }, rain);
Show($"[OBSERVATION] observe(drops=True) -> pleut-il ? {p_rain_if_drops}");
Show("  -> Le barometre predit la pluie : P(rain | drops) = True (correlation via storm)");

// NIVEAU 2 : intervention do(drops=True)
var do_drops = new InterventionalStatement();
do_drops.AddIntervention(drops, true);
do_drops.SetConclusion(rain);
bool p_rain_if_do_drops = reasoner.Query(ckb_bar, do_drops);
Show($"[INTERVENTION] do(drops=True)     -> pleut-il ? {p_rain_if_do_drops}");
Show("  -> Forcer le barometre NE FAIT PAS pleuvoir : P(rain | do(drops)) = False");


[OBSERVATION] observe(drops=True) -> pleut-il ? True

  -> Le barometre predit la pluie : P(rain | drops) = True (correlation via storm)

[INTERVENTION] do(drops=True)     -> pleut-il ? False

  -> Forcer le barometre NE FAIT PAS pleuvoir : P(rain | do(drops)) = False

**Visualisation de la chirurgie** : `do(drops=True)` remplace l'équation `drops <=> storm` par `drops <=> +` (constante).
Le lien `storm -> drops` est **rompu** — `drops` ne dépend plus de `storm`.

In [6]:
Show("SCM original :");
Show("  " + barometer.PrettyPrint().Replace("\n", "\n  "));

Show("\nSCM apres do(drops=True) :");
var barometer_do = barometer.Intervene(drops, true);
Show("  " + barometer_do.PrettyPrint().Replace("\n", "\n  "));
Show("\n=> L'equation 'drops <=> storm' est devenue 'drops <=> +' (constante).");
Show("   Le lien storm -> drops est ROMPU : drops ne depend plus de storm.");


SCM original :

    [fond]      storm
    [explique]  drops <=> storm
    [explique]  rain <=> storm


SCM apres do(drops=True) :

    [fond]      storm
    [explique]  drops <=> +
    [explique]  rain <=> storm


=> L'equation 'drops <=> storm' est devenue 'drops <=> +' (constante).

   Le lien storm -> drops est ROMPU : drops ne depend plus de storm.

## 4. Réseau Sprinkler — isoler l'effet causal propre

Un SCM plus riche : `cloudy` cause `sprinkler` et `rain` ; tous deux causent `wet`.
C'est lejouet classique pour montrer que `do(rain)` casse le confondeur `cloudy` tandis que `do(sprinkler)` préserve la causalité directe vers `wet`.

In [7]:
var cloudy = "cloudy"; var sprinkler = "sprinkler"; var rain2 = "rain"; var wet = "wet";

var sprinkler_net = new StructuralCausalModel();
sprinkler_net.AddBackgroundAtom(cloudy);
sprinkler_net.AddExplainableAtom(sprinkler, "cloudy");
sprinkler_net.AddExplainableAtom(rain2,    "cloudy");
sprinkler_net.AddExplainableAtom(wet, "sprinkler || rain");   // wet = sprinkler OR rain

Show("SCM reseau Sprinkler :");
Show(sprinkler_net.PrettyPrint());

var ckb_sp = new CausalKnowledgeBase(sprinkler_net);
Show("Base causale prete.");


SCM reseau Sprinkler :

  [fond]      cloudy
  [explique]  sprinkler <=> cloudy
  [explique]  rain <=> cloudy
  [explique]  wet <=> sprinkler || rain

Base causale prete.

In [8]:
Show("=== Correlation vs Causalite dans le reseau Sprinkler ===\n");

// (a) OBSERVATION : observe(rain=True) -> sprinkler ?
bool p_sp_if_rain = reasoner.Query(ckb_sp, new List<string> { rain2 }, sprinkler);
Show($"[OBS]  observe(rain=True)  -> sprinkler actif ? {p_sp_if_rain}");
Show("       (correlation : rain et sprinkler coincident via cloudy)\n");

// (b) INTERVENTION : do(rain=True) -> sprinkler ?
var do_rain = new InterventionalStatement();
do_rain.AddIntervention(rain2, true);
do_rain.SetConclusion(sprinkler);
bool p_sp_if_do_rain = reasoner.Query(ckb_sp, do_rain);
Show($"[INT]  do(rain=True)       -> sprinkler actif ? {p_sp_if_do_rain}");
Show("       (intervention : do(rain) CASSE cloudy->rain, donc plus de correlation avec sprinkler)\n");

// (c) INTERVENTION causale directe : do(sprinkler=True) -> wet ?
var do_sp = new InterventionalStatement();
do_sp.AddIntervention(sprinkler, true);
do_sp.SetConclusion(wet);
bool p_wet_if_do_sp = reasoner.Query(ckb_sp, do_sp);
Show($"[INT]  do(sprinkler=True)  -> herbe mouillee ? {p_wet_if_do_sp}");
Show("       (causalite directe preservee : sprinkler CAUSE wet)\n");

Show("=> P(sprinkler | observe rain) != P(sprinkler | do rain)");
Show("   Signature du do-calculus : l'intervention isole l'effet causal propre.");


=== Correlation vs Causalite dans le reseau Sprinkler ===


[OBS]  observe(rain=True)  -> sprinkler actif ? True

       (correlation : rain et sprinkler coincident via cloudy)


[INT]  do(rain=True)       -> sprinkler actif ? False

       (intervention : do(rain) CASSE cloudy->rain, donc plus de correlation avec sprinkler)


[INT]  do(sprinkler=True)  -> herbe mouillee ? True

       (causalite directe preservee : sprinkler CAUSE wet)


=> P(sprinkler | observe rain) != P(sprinkler | do rain)

   Signature du do-calculus : l'intervention isole l'effet causal propre.

## 5. Raisonnement contrefactuel — mondes possibles

Un **contrefactuel** : « on a observé `wet=True` ; si `sprinkler` avait été OFF, l'herbe aurait-elle tout de même été mouillée ? »

Sémantique des mondes possibles : on fixe le monde réel (observé), on applique l'intervention contrefactuelle
en gardant les variables exogènes **fixes**, puis on ré-évalue la conclusion dans le monde contrefactuel.

In [9]:
var cf = new CounterfactualStatement();
cf.AddObservation(wet, true);                  // monde reel : on a observe wet=True
cf.AddCounterfactualIntervention(sprinkler, false); // monde contrefactuel : sprinkler=False
cf.SetConclusion(wet);                         // question : wet dans le monde contrefactuel ?

bool result_cf = reasoner.Query(ckb_sp, cf);
Show($"[CONTREFACTUEL] observe(wet=True) ; 'si sprinkler avait ete OFF, herbe mouillee ?' {result_cf}");
Show("  -> Oui : sous cloudy=True, rain aurait mouille l'herbe meme sans arrosage.");


[CONTREFACTUEL] observe(wet=True) ; 'si sprinkler avait ete OFF, herbe mouillee ?' True

  -> Oui : sous cloudy=True, rain aurait mouille l'herbe meme sans arrosage.

## 6. Exercices

### Exercice 1 — Conclondant caché (ability / education / income)

Construisez un SCM où `ability` cause à la fois `education` et `income` (`ability` est le **conclondant**).
Comparez `observe(education=True) -> income` (biaisé par le conclondant) et `do(education=True) -> income` (effet causal propre).

> Indice : avec `ability` comme cause commune, l'observation `education` **porte** l'information sur `ability` (donc sur `income`),
> mais l'intervention `do(education)` coupe ce lien — l'effet direct devrait tomber à False.

In [10]:
#nullable enable
// Exercice 1 : a completer
var ability = "ability"; var education = "education"; var income = "income";

// TODO etudiant : construire le SCM edu_model
//   - ability  = atome de fond
//   - education = f(ability)
//   - income    = f(ability)   [ability est le confondeur !]
StructuralCausalModel? edu_model = null;  // TODO etudiant : new StructuralCausalModel() ...

// TODO etudiant : comparer observe(education) -> income  vs  do(education) -> income
bool? result_observe = null;  // TODO etudiant
bool? result_do      = null;  // TODO etudiant

Show("Exercice 1 a completer");
Show("  (attendu : observe(education) -> income True ; do(education) -> income False)");


Exercice 1 a completer

  (attendu : observe(education) -> income True ; do(education) -> income False)

### Exercice 2 — L'intervention ne remonte pas la causalité

Sur `ckb_sp`, comparez `do(cloudy=True) -> wet` (cause ancêtre, effet transmis) et `do(wet=True) -> rain`.
Le graphe est **orienté** : forcer un effet ne propage rien vers ses causes.

In [11]:
#nullable enable
// Exercice 2 : a completer
// TODO etudiant : construire 2 InterventionalStatement et interroger ckb_sp
// Indice : do(wet=True) sur un EFFET ne remonte pas la causalite (le graphe est oriente).

InterventionalStatement? ex2_a = null;  // TODO etudiant : do(cloudy=True) -> wet
InterventionalStatement? ex2_b = null;  // TODO etudiant : do(wet=True)    -> rain
Show("Exercice 2 a completer");


Exercice 2 a completer

### Exercice 3 — Contrefactuel sur la cause

Construisez un `CounterfactualStatement` sur `ckb_sp` : `observe(rain=True)` ; intervention contrefactuelle `cloudy=False` ;
conclusion `rain`. Réfléchissez : `cloudy` est la **cause** de `rain` — si `cloudy` avait été faux, `rain` aurait-il eu lieu ?

In [12]:
#nullable enable
// Exercice 3 : a completer
// TODO etudiant : CounterfactualStatement sur ckb_sp
//   observe(rain=True) ; cf-intervention cloudy=False ; conclusion = rain
CounterfactualStatement? cf_ex3 = null;  // TODO etudiant
Show("Exercice 3 a completer");
Show("  (reflechir : cloudy est la cause de rain ; si cloudy avait ete faux, rain n'aurait pas eu lieu)");


Exercice 3 a completer

  (reflechir : cloudy est la cause de rain ; si cloudy avait ete faux, rain n'aurait pas eu lieu)

---

## Synthèse

Ce twin C# a réimplémenté **from-scratch** les trois piliers de l'inférence causale de Pearl :

| Requête | Sémantique implémentée |
|--------|------------------------|
| `observe(X)` | filtrer les mondes possibles consistants |
| `do(X)` | **chirurgie de graphe** : remplacer l'équation de X par une constante |
| contrefactuel | monde réel observé + surgery sur fond **fixe** |

La lib Tweety `causal-1.30.jar` (reasoner **argumentatif**) fournit la même sémantique en boîte noire ;
l'implémentation from-scratch rend **visible** pourquoi l'intervention isole l'effet causal propre — la valeur pédagogique.

**Voir aussi** : [Tweety-11-Causal.ipynb](Tweety-11-Causal.ipynb) (Python + Tweety jar), marathon #4956, EPIC #3801.

### Pour aller plus loin

- **[Notebook-pont — du graphe causal au do-calculus](../../Probas/DecisionTheory/Causal-Bridges/Do-Calculus-Bridge.ipynb)** : la chirurgie de graphe implémentée from-scratch ici (remplacer l'équation de `X` par une constante) est la **sémantique** du do-operator ; le pont en montre le **calcul** — l'échelle de causalité et les 3 règles de Pearl exécutées sur l'outil de référence `dowhy` (identification backdoor/front-door + estimation + réfutation), en **quantitatif** là où ce twin décide en booléen. Il relie cette série à Infer-5, PyMC-5 et ICT-5.